In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [47]:
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep
from scipy.stats import norm

In [2]:
df= pd.read_csv('marketing_campaign.csv')
df.head()

,date,channel,platform,campaign_type,campaign_id,campaign_name,ab_variant,geo,device,audience,...,impressions,clicks,sessions,conversions,revenue_gbp,flag_blank_revenue,spend_gbp,flag_blank_spend,zero_spend_with_activity,revenue_without_spend
0,11/11/2025,Display,DV360,Free Trial,C51468,Trial_CompetitorTerms,B,UK,All,CRM,...,286855,1028,1028,43,1654.95,0,2010.52,0,0,0
1,02/11/2025,Paid Search,Google Ads,Free Trial,C06687,Trial_CompetitorTerms,B,UK,All,CRM,...,37431,1020,1006,86,3273.09,0,679.96,0,0,0
2,04/11/2025,Affiliate,Partner Network,Free Trial,C00918,FreeTrial_NonBrand,A,IE,Mobile,CRM,...,2043,100,84,7,188.28,0,27.23,0,0,0
3,11/11/2025,Paid Social,LinkedIn Ads,Free Trial,C89894,Trial_Reengagement,B,IE,Mobile,CRM,...,25616,431,409,16,569.32,0,NaN,1,1,1
4,27/11/2025,Paid Social,Meta Ads,Discount Offer,C77790,Discount_20pct,A,UK,All,Retargeting,...,82394,698,589,29,1650.35,0,498.73,0,0,0


## 1. Filter Data

In [28]:
camp_data = df[(df.campaign_type == 'Discount Offer') & (df.channel == 'Paid Social') & (df.campaign_name == 'Discount_Flash48h')]
camp_data.head()

,date,channel,platform,campaign_type,campaign_id,campaign_name,ab_variant,geo,device,audience,...,impressions,clicks,sessions,conversions,revenue_gbp,flag_blank_revenue,spend_gbp,flag_blank_spend,zero_spend_with_activity,revenue_without_spend
5,08/11/2025,Paid Social,LinkedIn Ads,Discount Offer,C51603,Discount_Flash48h,B,UK,All,Prospecting,...,101969,1020,1020,78,7134.70,0,734.58,0,0,0
8,19/11/2025,Paid Social,Meta Ads,Discount Offer,C47476,Discount_Flash48h,B,IE,All,Prospecting,...,15365,155,125,12,387.14,0,152.05,0,0,0
12,23/11/2025,Paid Social,Meta Ads,Discount Offer,C47476,Discount_Flash48h,B,IE,NaN,Retargeting,...,14041,206,193,42,1624.82,0,119.99,0,0,0
28,02/11/2025,Paid Social,LinkedIn Ads,Discount Offer,C51603,Discount_Flash48h,A,IE,All,Retargeting,...,36993,740,740,73,4147.39,0,489.26,0,0,0
29,20/11/2025,Paid Social,Meta Ads,Discount Offer,C47476,Discount_Flash48h,B,UK,Mobile,CRM,...,149602,3635,3635,303,13256.86,0,1192.45,0,0,0


In [29]:
var_a = camp_data[camp_data['ab_variant']=='A']
var_a.head()

,date,channel,platform,campaign_type,campaign_id,campaign_name,ab_variant,geo,device,audience,...,impressions,clicks,sessions,conversions,revenue_gbp,flag_blank_revenue,spend_gbp,flag_blank_spend,zero_spend_with_activity,revenue_without_spend
28,02/11/2025,Paid Social,LinkedIn Ads,Discount Offer,C51603,Discount_Flash48h,A,IE,All,Retargeting,...,36993,740,740,73,4147.39,0,489.26,0,0,0
41,12/11/2025,Paid Social,TikTok Ads,Discount Offer,C26685,Discount_Flash48h,A,UK,Desktop,Retargeting,...,138850,1504,1504,282,8371.27,0,1329.47,0,0,0
82,30/11/2025,Paid Social,TikTok Ads,Discount Offer,C26685,Discount_Flash48h,A,IE,All,Prospecting,...,0,0,0,0,0.00,0,32.27,0,0,0
98,11/11/2025,Paid Social,Meta Ads,Discount Offer,C47476,Discount_Flash48h,A,IE,All,CRM,...,38992,479,407,48,2027.40,0,419.68,0,0,0
121,23/11/2025,Paid Social,Meta Ads,Discount Offer,C47476,Discount_Flash48h,A,UK,Desktop,Retargeting,...,94886,761,761,96,4759.06,0,864.93,0,0,0


In [30]:
var_b = camp_data[camp_data['ab_variant']=='B']
var_b.head()

,date,channel,platform,campaign_type,campaign_id,campaign_name,ab_variant,geo,device,audience,...,impressions,clicks,sessions,conversions,revenue_gbp,flag_blank_revenue,spend_gbp,flag_blank_spend,zero_spend_with_activity,revenue_without_spend
5,08/11/2025,Paid Social,LinkedIn Ads,Discount Offer,C51603,Discount_Flash48h,B,UK,All,Prospecting,...,101969,1020,1020,78,7134.70,0,734.58,0,0,0
8,19/11/2025,Paid Social,Meta Ads,Discount Offer,C47476,Discount_Flash48h,B,IE,All,Prospecting,...,15365,155,125,12,387.14,0,152.05,0,0,0
12,23/11/2025,Paid Social,Meta Ads,Discount Offer,C47476,Discount_Flash48h,B,IE,NaN,Retargeting,...,14041,206,193,42,1624.82,0,119.99,0,0,0
29,20/11/2025,Paid Social,Meta Ads,Discount Offer,C47476,Discount_Flash48h,B,UK,Mobile,CRM,...,149602,3635,3635,303,13256.86,0,1192.45,0,0,0
38,03/11/2025,Paid Social,TikTok Ads,Discount Offer,C26685,Discount_Flash48h,B,UK,Mobile,CRM,...,263561,4385,3288,489,23024.12,0,2949.58,0,0,0


# 2. Check for bias

In [36]:
#check number od rows in each variant type
rows_var_a = var_a.shape[0]
rows_var_b = var_b.shape[0]
print('Rows in A : ', rows_var_a, '\nRows in B : ', rows_var_b)

Rows in A :  417 
Rows in B :  419


In [43]:
#check for the total number of sessions
var_a_sess = sum(var_a.sessions)
var_b_sess = sum(var_b.sessions)
print('Total sessions A:',var_a_sess , '\nTotal sessions B:', var_b_sess )
total_traffic = var_a_sess + var_b_sess
print('Traffic percentage for Variant A:', round((var_a_sess/total_traffic)*100,2))
print('Traffic percentage for Variant B:', round((var_b_sess/total_traffic)*100,2))

Total sessions A: 404714 
Total sessions B: 361506
Traffic percentage for Variant A: 52.82
Traffic percentage for Variant B: 47.18


### About 5% difference in traffic is observed. The dataset does not show any high imbalances or biases regarding traffic

In [44]:
#check for the total number of conversiosn
var_a_conv = sum(var_a.conversions)
var_b_conv = sum(var_b.conversions)
print('Total conversions A:',var_a_conv , '\nTotal conversions B:', var_b_conv )

Total conversions A: 42467 
Total conversions B: 37681


###  Variant B has slightly lesser conversions than variant A, this could be explained by the traffic volume and by performance alone. Critical Bias is not detected, the traffic imbalance is realistic and acceptable. 

### Traffic allocation across variants was moderately imbalanced, with Variant A receiving a slightly higher share of sessions.

In [45]:
# calculate CVR
cvr_a = (sum(var_a.conversions)/sum(var_a.sessions) )*100
cvr_b = (sum(var_b.conversions)/sum(var_b.sessions) )*100
print('Conversion rate of variant A: ', round(cvr_a,2) ,'\nConversion rate of variant B ', round(cvr_b,2))



Conversion rate of variant A:  10.49 
Conversion rate of variant B  10.42


# 3. Hypothesis Testing

One-sided test: Variant B > Variant A

proportions_ztest tests (p1 - p2).
So if we want B > A, we pass [B, A] in that order and use alternative='larger'.

Null hypothesis (H₀) : Variant B’s conversion rate is less than or equal to Variant A’s conversion rate.
Alternative hypothesis (H₁) :Variant B’s conversion rate is greater than Variant A’s conversion rate.


z = (p1-p2)/SE

In [51]:
#num_success will contain the conversions of B and A variant respectively
alpha = 0.05
num_success = np.array([
    var_b.conversions.sum(),
    var_a.conversions.sum()
])

#num_obs will contain the sessions of B and A variant respectively
num_obs = np.array([
    var_b.sessions.sum(),
    var_a.sessions.sum()
])

z_stat, p_value = proportions_ztest(num_success, num_obs, alternative='larger')

print("z-stat:", z_stat)
print("p-value:", p_value)

if p_value < alpha:
    print('Reject H0, variant B conversion is greater than A')
else:
    print('Fail to reject H0, Variant B conversion is less than or equal to Variant A')

z-stat: -0.995896513211333
p-value: 0.8403497851783728
Fail to reject H0, Variant B conversion is less than or equal to Variant A


### since p-value is greater than alpha, The test fails to provide evidence that Variant B has a higher conversion rate than Variant A.. Furthermore Variant B performs slightly worse, but the difference is small and statistically indistinguishable from noise.

# 4. Calculate the lift

In [53]:
cvr_b = num_success[0] / num_obs[0]
cvr_a = num_success[1] / num_obs[1]

abs_lift = cvr_b - cvr_a
rel_lift = abs_lift / cvr_a

print("CVR A:", cvr_a)
print("CVR B:", cvr_b)
print("Absolute lift (B-A):", abs_lift)
print("Relative lift:", rel_lift)


CVR A: 0.10493088946762405
CVR B: 0.10423340138199642
Absolute lift (B-A): -0.0006974880856276261
Relative lift: -0.006647118776619471


## Variant B exhibited a slightly lower conversion rate than Variant A (−0.07 percentage points). A one-sided two-proportion z-test failed to reject the null hypothesis (p = 0.84), indicating no statistically significant evidence that Variant B improves conversion performance. The observed difference is small in magnitude and likely attributable to random variation.